In [0]:
%pip install datasets==3.2.0 huggingface_hub==0.26.5

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
from datasets import load_dataset

dataset = load_dataset("lighteval/lexglue", "ledgar")
dataset

/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:56: UserWarning: The cache_dir for this dataset is /tmp/.hf.data.cache, which is not a persistent path.Therefore, if/when the cluster restarts, the downloaded dataset will be lost.The persistent storage options for this workspace/cluster config are: [UC Volumes].Please update either `cache_dir` or the environment variable `HF_DATASETS_CACHE`to be under one of the following root directories: ['/Volumes/']
  warnings.warn(warning_message)
/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:24: UserWarning: During large dataset downloads, there could be multiple progress bar widgets that can cause performance issues for your notebook or browser. To avoid these issues, use `datasets.utils.logging.disable_progress_bar()` to turn off the progress bars.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 60000
    })
    validation: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 10000
    })
})

In [0]:
print(dataset)
print(dataset["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 60000
    })
    validation: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['input', 'references', 'gold'],
        num_rows: 10000
    })
})
['input', 'references', 'gold']


In [0]:
from pyspark.sql import functions as F

spark_dfs = []

for split_name in dataset.keys():
    pdf = dataset[split_name].to_pandas()
    sdf = spark.createDataFrame(pdf)
    sdf = sdf.withColumn("split", F.lit(split_name))
    spark_dfs.append(sdf)

ledgar_df = spark_dfs[0]

for sdf in spark_dfs[1:]:
    ledgar_df = ledgar_df.unionByName(sdf)

display(ledgar_df.limit(10))
ledgar_df.printSchema()

input,references,gold,split
"Except as otherwise set forth in this Debenture, the Company, for itself and its legal representatives, successors and assigns, expressly waives presentment, protest, demand, notice of dishonor, notice of nonpayment, notice of maturity, notice of protest, presentment for the purpose of accelerating maturity, and diligence in collection.","List(Qualifications, Modifications, Titles, Authority, Effective Dates, Counterparts, Agreements, Releases, Brokers, No Defaults, Severability, Authorizations, Integration, Terms, Insurances, Transactions With Affiliates, Indemnifications, Expenses, Organizations, Disability, Jurisdictions, Records, Further Assurances, Duties, Submission To Jurisdiction, Payments, Vacations, Assigns, Enforcements, Sanctions, Waivers)",List(Waivers),train
"No ERISA Event has occurred or is reasonably expected to occur that, when taken together with all other such ERISA Events for which liability is reasonably expected to occur, could reasonably be expected to result in a Material Adverse Effect. Neither Borrower nor any ERISA Affiliate maintains or contributes to or has any obligation to maintain or contribute to any Multiemployer Plan or Plan, nor otherwise has any liability under Title IV of ERISA.","List(Interests, Enforcements, No Conflicts, Consents, Approvals, Applicable Laws, Publicity, Venues, Binding Effects, Costs, Payments, Participations, Liens, Disability, Intellectual Property, Sales, Amendments, Counterparts, Agreements, Headings, No Waivers, Existence, Anti-Corruption Laws, General, Brokers, Tax Withholdings, Enforceability, Financial Statements, Waivers, Cooperation, Erisa)",List(Erisa),train
"This Amendment may be executed by one or more of the parties hereto on any number of separate counterparts, and all of said counterparts taken together shall be deemed to constitute one and the same instrument. This Amendment may be delivered by facsimile or other electronic transmission of the relevant signature pages hereof.","List(Warranties, Releases, Interests, Subsidiaries, Enforcements, Qualifications, Entire Agreements, Authorizations, Effective Dates, Closings, Compliance With Laws, Expenses, Construction, Notices, General, Binding Effects, Approvals, Payments, Positions, Severability, Cooperation, Confidentiality, Agreements, Venues, Non-Disparagement, Jurisdictions, No Defaults, Survival, Effectiveness, Sales, Counterparts)",List(Counterparts),train
"From time to time, as and when required by the Surviving Corporation or by its successors or assigns, there shall be executed and delivered on behalf of Ashford (DE) such deeds and other instruments, and there shall be taken or caused to be taken by it all such further and other action, as shall be appropriate, advisable or necessary in order to vest, perfect or confirm, of record or otherwise, in the Surviving Corporation the title to and possession of all property, interests, assets, rights, privileges, immunities, powers, franchises and authority of Ashford (DE), and otherwise to carry out the purposes of this Agreement. The officers and directors of the Surviving Corporation are fully authorized in the name and on behalf of Ashford (DE) or otherwise, to take any and all such action and to execute and deliver any and all such deeds and other instruments.","List(Warranties, Books, Qualifications, Publicity, Non-Disparagement, Amendments, Forfeitures, Base Salary, Benefits, Miscellaneous, Agreements, Financial Statements, Representations, Subsidiaries, Entire Agreements, Interpretations, Positions, Employment, Headings, Authority, Tax Withholdings, Waiver Of Jury Trials, Change In Control, Waivers, No Conflicts, General, Litigations, Indemnity, Anti-Corruption Laws, Consent To Jurisdiction, Further Assurances)",List(Further Assurances),train
"Commencing March 7, 2016 and during the Employment Period, the Company shall pay to the Executive a base salary at the rate of no less than $750,000 per calendar year (the “Base 

root
 |-- input: string (nullable = true)
 |-- references: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- gold: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- split: string (nullable = false)



In [0]:
ledgar_clean_df = (
    ledgar_df
    .select(
        F.col("input").alias("provision_text"),
        F.col("gold").alias("category_label"),
        F.col("split")
    )
    .withColumn("provision_id", F.monotonically_increasing_id())
    .withColumn("text_length", F.length("provision_text"))
    .withColumn("ingested_at", F.current_timestamp())
)

display(ledgar_clean_df.limit(10))

provision_text,category_label,split,provision_id,text_length,ingested_at
"Except as otherwise set forth in this Debenture, the Company, for itself and its legal representatives, successors and assigns, expressly waives presentment, protest, demand, notice of dishonor, notice of nonpayment, notice of maturity, notice of protest, presentment for the purpose of accelerating maturity, and diligence in collection.",List(Waivers),train,0,338,2026-06-05T19:52:58.741Z
"No ERISA Event has occurred or is reasonably expected to occur that, when taken together with all other such ERISA Events for which liability is reasonably expected to occur, could reasonably be expected to result in a Material Adverse Effect. Neither Borrower nor any ERISA Affiliate maintains or contributes to or has any obligation to maintain or contribute to any Multiemployer Plan or Plan, nor otherwise has any liability under Title IV of ERISA.",List(Erisa),train,1,452,2026-06-05T19:52:58.741Z
"This Amendment may be executed by one or more of the parties hereto on any number of separate counterparts, and all of said counterparts taken together shall be deemed to constitute one and the same instrument. This Amendment may be delivered by facsimile or other electronic transmission of the relevant signature pages hereof.",List(Counterparts),train,2,328,2026-06-05T19:52:58.741Z
"From time to time, as and when required by the Surviving Corporation or by its successors or assigns, there shall be executed and delivered on behalf of Ashford (DE) such deeds and other instruments, and there shall be taken or caused to be taken by it all such further and other action, as shall be appropriate, advisable or necessary in order to vest, perfect or confirm, of record or otherwise, in the Surviving Corporation the title to and possession of all property, interests, assets, rights, privileges, immunities, powers, franchises and authority of Ashford (DE), and otherwise to carry out the purposes of this Agreement. The officers and directors of the Surviving Corporation are fully authorized in the name and on behalf of Ashford (DE) or otherwise, to take any and all such action and to execute and deliver any and all such deeds and other instruments.",List(Further Assurances),train,3,870,2026-06-05T19:52:58.741Z
"Commencing March 7, 2016 and during the Employment Period, the Company shall pay to the Executive a base salary at the rate of no less than $750,000 per calendar year (the “Base Salary”), less applicable deductions, and prorated for any partial month or year, as applicable. The Base Salary shall be reviewed for increase by the Compensation Committees of AFG and AAC (the “Compensation Committees”) no less frequently than annually and may be increased in the discretion of the Compensation Committees. Any such adjusted Base Salary shall constitute the “Base Salary” for purposes of this Agreement. The Base Salary shall be paid in substantially equal installments in accordance with AAC’s regular payroll procedures. The Executive’s Base Salary may not be decreased during the Employment Period. The Company shall provide the Executive with a payment in an amount equal to the difference between (i) the Base Salary payments the Executive would have received had he been paid at the rate set forth in this Section 4(a) during the period commencing on March 7, 2016 and ending on the Effective Date hereof and (ii) the actual salary payments made to the Executive during such period, payable in a lump sum on a regular payroll date as soon as practicable following the Effective Date.",List(Base Salary),train,4,1286,2026-06-05T19:52:58.741Z
"All notices required or permitted under this Agreement will be in writing, will reference this Agreement, and will be deemed given: (i) when delivered personally; (ii) one (1) business day after deposit with a nationally-recognized express courier, with written confirmation of receipt; or (iii) three (3) business days after having been sent by registered or cert

In [0]:
table_name = "default.ledgar_lexglue"

(
    ledgar_clean_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

In [0]:
df = spark.table("default.ledgar_lexglue")

display(df.limit(10))
df.printSchema()

provision_text,category_label,split,provision_id,text_length,ingested_at
"This Agreement may be executed and delivered by facsimile signature and in two or more counterparts, each of which shall be deemed an original, but all of which together shall constitute one and the same instrument. Counterpart signature pages to this Agreement transmitted by facsimile transmission, by electronic mail in “portable document format” (“pdf”) form, or by any other electronic means intended to preserve the original graphic and pictorial appearance of a document, will have the same effect as physical delivery of the paper document bearing an original signature.",List(Counterparts),train,20000,579,2026-06-05T19:53:17.562Z
"Each of the Parties will, at their own respective expense (and not subject to cost sharing hereunder) procure and maintain during the Term, insurance policies adequate to cover their obligations hereunder and consistent with the normal business practices of prudent biopharmaceutical companies of similar size and scope (or reasonable self-insurance sufficient to provide materially the same level and type of protection). Such insurance will not create a limit to either Party’s liability hereunder.",List(Insurances),train,20001,501,2026-06-05T19:53:17.562Z
Headings in this letter are for reference only and shall not be deemed to have any substantive effect.,List(Headings),train,20002,102,2026-06-05T19:53:17.562Z
"Subject to compliance with the provisions of this Agreement (including Section 5.1(b)(iv) ), the Board shall have the right to cause the LLC to authorize, designate, issue or sell to any Person (including Unitholders and Affiliates) any additional Equity Securities (which for purposes of this Agreement shall be “ Additional Securities ”). Subject to the provisions of this Agreement, including Section 5.1(b)(iv) , the Board shall determine the terms and conditions governing the issuance of such Additional Securities, including the number and designation of such Additional Securities, the designations, preferences (with respect to distributions, liquidations, or otherwise) over any other Units and relative, participating, optional or other special rights, powers and duties, including rights, powers and duties senior or junior to, or pari passu with, any other Units, any required contributions in connection therewith and voting rights. Subject to Section 5.1(b)(iv) and Section 15.3 , the Board shall, in its sole discretion, be permitted to amend this Agreement in connection with the authorization, designation reservation or issuance of any Additional Securities. Any Person who acquires Units may be admitted to the LLC as a Unitholder pursuant to the terms of Section 11.2 hereof. If any Person acquires additional Units or other interests in the LLC or is admitted to the LLC as an Additional Unitholder, the LLC shall amend Schedule A to reflect such additional issuance and/or Unitholder, as the case may be.",List(Interests),train,20003,1527,2026-06-05T19:53:17.562Z
"Except for disputes regarding obligations that admit, forthwith, judicial execution, the Parties undertake to endeavor best efforts to amicably resolve by mutual negotiation any disputes arising from or related to this Agreement and/or its Exhibits/Schedules and/or related thereto, including but not limited to any issues relating to the existence, validity, effectiveness, termination or contractual performance (“ Dispute ”). In case such mutual agreement is not reached, any Dispute will be referred to and exclusively and finally settled by binding arbitration according to the rules (“ Arbitration Rules ”) of the Arbitration and Mediation Center of the Brazil-Canada Chamber of Commerce (“ Arbitration Chamber ”), which rules are deemed to be incorporated by reference to this Agreement, except as such Arbitration Rules may be modified herein or by mutual agreement by the Parties.",List(Arbitration),train,20004,889,2026-06-05T19:53:17.562Z
"The Employee shall be enti

root
 |-- provision_text: string (nullable = true)
 |-- category_label: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- split: string (nullable = true)
 |-- provision_id: long (nullable = true)
 |-- text_length: integer (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [0]:
print("Total rows:", df.count())

Total rows: 80000


In [0]:
display(
    df.groupBy("split")
      .count()
      .orderBy("split")
)

split,count
test,10000
train,60000
validation,10000


In [0]:
display(
    df.groupBy("category_label")
      .count()
      .orderBy(F.desc("count"))
)

category_label,count
List(Governing Laws),4243
List(Counterparts),3346
List(Notices),3313
List(Entire Agreements),3105
List(Severability),2552
List(Survival),1951
List(Amendments),1948
List(Assignments),1730
List(Expenses),1577
List(Terms),1511


In [0]:
display(
    df.agg(F.countDistinct("category_label").alias("distinct_category_labels"))
)

distinct_category_labels
100


In [0]:
display(
    df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in ["provision_text", "category_label", "split"]
    ])
)

provision_text,category_label,split
0,0,0


In [0]:
display(
    df.filter(F.trim(F.col("provision_text")) == "")
)

provision_text,category_label,split,provision_id,text_length,ingested_at


In [0]:
display(
    df.groupBy("provision_text")
      .count()
      .filter(F.col("count") > 1)
      .orderBy(F.desc("count"))
)

provision_text,count


## Data Quality Summary

The LEDGAR subset of LexGLUE was successfully ingested from Hugging Face and stored as a Delta Table (`default.ledgar_lexglue`) within Databricks.

### Validation Results
- Total records loaded: 80,000
- Train, validation, and test splits were successfully preserved and verified
- Label distribution was analyzed across all categories
- 100 distinct legal provision categories were identified
- Null value checks were performed on provision text, category labels, references, and split fields
- Empty text records were evaluated
- Duplicate provision records were reviewed

### Data Quality Observations
- The dataset is well-structured and requires minimal preprocessing prior to downstream NLP workflows.
- Legal provision text and category labels are consistently populated across records.
- Category frequencies are imbalanced, which is expected in legal-domain classification datasets where certain provision types occur more frequently than others.
- The dataset consists of publicly available SEC EDGAR filings and does not contain confidential client intake information, minimizing privacy and compliance concerns.
- The dataset is suitable for legal text classification, retrieval-augmented generation (RAG), and instruction fine-tuning experiments within the scope of this project.